> ## SOLUTION / ANSWER KEY &mdash; Lab 6.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-langgraph-state-machine.ipynb`](../lab-08-langgraph-state-machine.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.8 &mdash; LangGraph: a State Graph with Human Approval

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Route each step to a node: tool, human-approval, or end
- Wire a real LangGraph StateGraph with conditional edges
- Run a sequence through the compiled graph and read the node path

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; LangGraph -- agents as state graphs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-08"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
When one implicit loop isn't enough, **LangGraph** models the agent as a **state graph**: **nodes** are
steps, **edges** are transitions you control (deck slides 12&ndash;13). A key win is a **human approval**
node: a **risky** action (send email, delete, pay) is routed to a human before it runs, while safe actions
go straight to the tool node. Here you wire a **real** `StateGraph` and run it &mdash; the graph really
compiles and executes (no LLM needed for this one).

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class GState(TypedDict):
    actions: list
    path: list

print("nodes we will wire: tool, human, END |", "END sentinel:", END)

## Build it
Complete `route` (which node an action goes to) and the conditional entry point, then compile and run the
real graph. A risky step lands on the human node before anything happens.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

RISKY = {"send_email", "delete", "pay"}

class GState(TypedDict):
    actions: list
    path: list

def route(state):
    done_so_far = len(state['path'])
    action = state['actions'][done_so_far] if done_so_far < len(state['actions']) else 'done'
    if action == "done":
        return "end"
    if action in RISKY:
        return "human"
    return "tool"          # any safe tool call

def tool_node(state):  return {'path': state['path'] + ['tool']}
def human_node(state): return {'path': state['path'] + ['human']}

def build_graph():
    g = StateGraph(GState)
    g.add_node('tool', tool_node)
    g.add_node('human', human_node)
    edges = {'tool': 'tool', 'human': 'human', 'end': END}
    g.set_conditional_entry_point(route, edges)
    g.add_conditional_edges('tool', route, edges)
    g.add_conditional_edges('human', route, edges)
    return g.compile()

def run_graph(actions):
    return build_graph().invoke({'actions': actions, 'path': []})['path']

try:
    print("search      ->", route({"actions": ["search"], "path": []}))
    print("send_email  ->", route({"actions": ["send_email"], "path": []}))
    print("safe then risky :", run_graph(["search", "send_email", "done"]))
    print("delete is gated :", run_graph(["search", "delete"]))
    print("all safe        :", run_graph(["search", "search", "done"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The path shows **`human`** appearing exactly where a risky action was &mdash; the graph *paused* for approval.
- `set_conditional_entry_point` + `add_conditional_edges` are the real LangGraph API; this graph genuinely compiled and ran.
- LangChain gets an agent running; LangGraph puts its flow under explicit control.

## Your turn (open task &mdash; no grader)
Add a new risky action name to `RISKY` (e.g. `"wire_transfer"`) and a sequence that uses it, then re-run.
**What good looks like:** your new risky action routes to `human` while safe actions go straight to `tool`.
(Advanced: add a real `llm`-backed node that decides the next action &mdash; the graph will call it.)

---
### Key takeaway
LangChain gets an agent running; LangGraph puts its flow under control -- explicit nodes, branching, and a first-class human-approval pause. Start simple; graduate when you need it.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>